In [1]:
import os
import sys
from pathlib import Path
PROJECT_ROOT = Path().resolve().parent
sys.path.append(str(PROJECT_ROOT))

In [2]:
from rag.retrieval import retrieval
from rag.vector_database import VectorDatabase

c:\Users\hafsa\Desktop\Islamic-AI-Assistant\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 391/391 [00:03<00:00, 119.07it/s]


In [3]:
import pandas as pd

data_path = "../../data/evaluation/eval_dataset_batch2.csv"
doc_path ="../../data/gold"
data = pd.read_csv(data_path)
vector_db = VectorDatabase(
        collection_name="quran_vdb",
        path=doc_path,
    )

data = data.assign(
    parent_id_gt=lambda df_: (
        df_["surah_n"].astype(str)
        + ":"
        + df_["ayah_n"].astype(str)
    )
)
k_param =  [1, 3, 5, 7, 10, 15]

for k in k_param:
    col_name = f"result_{k}"
    col_check = f"check_{k}"

    # create empty column
    data[col_name] = ""

    for i, row in data.iterrows():

        results, parent_ids = retrieval(
            docs_path=os.path.join(doc_path, "quran_docs.pkl"),
            query=row["question"],
            vector_db=vector_db,
            top_k=k,
            return_parents_ids=True,
        )
        data.loc[i,col_check] = data.loc[i,"parent_id_gt"] in parent_ids
        data.loc[i, col_name] = ",".join(parent_ids)

data.to_csv("../../data/evaluation/eval_data_final.csv",index=False)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 461.06it/s] 


STEP 1: Loading QuranDoc store
Docs path: ../../data/gold\quran_docs.pkl
Successfully loaded 10 QuranDoc objects.

STEP 2: Performing semantic search
Query: What does the tafsir say grieves the hypocrites?
Top K: 1
Search completed successfully.

STEP 3: Inspecting raw search results
Number of retrieved chunks: 1

Result 1
  Chunk ID : 2:121:31
  Surah    : 2
  Ayah     : 121
  Distance : 0.45493972301483154

STEP 4: Extracting parent IDs
Unique parent IDs (1):
  - 2:121

STEP 5: Building final context
Processing parent_id: 2:121
  -> Found Surah 2, Ayah 121
  -> Tafsir length: 8076 characters

STEP 6: Final summary
Documents included in context: 1
Final context length: 8453 characters
STEP 1: Loading QuranDoc store
Docs path: ../../data/gold\quran_docs.pkl
Successfully loaded 10 QuranDoc objects.

STEP 2: Performing semantic search
Query: What do the hypocrites say when calamity strikes the Prophet?
Top K: 1
Search completed successfully.

STEP 3: Inspecting raw search results
Number 

### Read the data

In [4]:
import plotly.express as px
import pandas as pd
data = pd.read_csv("../../data/evaluation/eval_data_final.csv")

### Calculate metrics: Recall@K

In [5]:
rows = []

for col in data.columns:
    if col.startswith("check"):
        rows.append([
            col.split('_')[1],
            data[col].sum() / len(data)
        ])

data_benchmark = pd.DataFrame(rows, columns=['value_k', 'value'])
data["recall_1"] = data["check_1"].astype(int)
data["recall_3"] = data["check_3"].astype(int)
data["recall_5"] = data["check_5"].astype(int)
data["recall_7"] = data["check_7"].astype(int)
data["recall_10"] = data["check_10"].astype(int)
data["recall_15"] = data["check_15"].astype(int)

In [6]:

recall_cols = ["recall_1","recall_3","recall_5","recall_7","recall_10","recall_15"]

mean_scores = data[recall_cols].mean()

df_plot = pd.DataFrame({
    "K": [1, 3, 5, 7, 10, 15],
    "Recall": mean_scores.values*100
})
fig = px.line(
    df_plot,
    x="K",
    y="Recall",
    markers=True,
    title="Retrieval Performance: Recall@K (%) vs K",
)

fig.update_layout(
    xaxis_title="K (Top-K retrieved documents)",
    yaxis_title="Recall@K (%)",
    template="simple_white"
)


fig.show()

### Calculate metrics: MRR

In [7]:
for col in data.columns:

    if col.startswith("result_"):

        k = col.split("_")[1]
        mrr_col = f"mrr_{k}"

        def compute_mrr(row):

            gt = row["parent_id_gt"]

            retrieved = row[col]

            # handle empty retrieval
            if pd.isna(retrieved) or retrieved == "":
                return 0.0

            retrieved_list = retrieved.split(",")

            # search for first correct occurrence
            for rank, pid in enumerate(retrieved_list, start=1):

                if pid.strip() == gt:
                    return 1 / rank

            return 0.0

        data[mrr_col] = data.apply(compute_mrr, axis=1)

In [8]:

mrr_cols = ["mrr_1","mrr_3","mrr_5","mrr_7","mrr_10","mrr_15"]

mean_scores = data[mrr_cols].mean()

df_plot = pd.DataFrame({
    "K": [1, 3, 5, 7, 10, 15],
    "mrr": mean_scores.values*100
})
fig = px.line(
    df_plot,
    x="K",
    y="mrr",
    markers=True,
    title="Retrieval Performance: mrr@K (%) vs K",
)

fig.update_layout(
    xaxis_title="K (Top-K retrieved documents)",
    yaxis_title="mrr@K (%)",
    template="simple_white"
)

fig.show()

In [13]:
import plotly.graph_objects as go

k_values = [1, 3, 5, 7, 10, 15]

# ---------------------------------------------------
# Recall
# ---------------------------------------------------
recall_cols = [
    "recall_1",
    "recall_3",
    "recall_5",
    "recall_7",
    "recall_10",
    "recall_15"
]

recall_scores = data[recall_cols].mean().values

# ---------------------------------------------------
# MRR
# ---------------------------------------------------
mrr_cols = [
    "mrr_1",
    "mrr_3",
    "mrr_5",
    "mrr_7",
    "mrr_10",
    "mrr_15"
]

mrr_scores = data[mrr_cols].mean().values

# ---------------------------------------------------
# Figure
# ---------------------------------------------------
fig = go.Figure()

# Recall curve
fig.add_trace(
    go.Scatter(
        x=k_values,
        y=recall_scores,
        mode="lines+markers",
        name="Recall@K"
    )
)

# MRR curve
fig.add_trace(
    go.Scatter(
        x=k_values,
        y=mrr_scores,
        mode="lines+markers",
        name="MRR@K"
    )
)

# ---------------------------------------------------
# Layout
# ---------------------------------------------------
fig.update_layout(
    title="Retrieval Performance vs K",
    xaxis_title="K (Top-K Retrieved Documents)",
    yaxis_title="Score",
    template="simple_white",
    legend_title="Metric",
)

fig.update_yaxes()

fig.show()